# RetailMind AI ? Day 5: Neural Network for Churn

This notebook has the complete code for data preparation, training, testing, and saving. It also trains a Random Forest so you can compare a neural network with a familiar model.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
csv_names = ["Telco-Customer-Churn.csv", "WA_Fn-UseC_-Telco-Customer-Churn.csv", "telco_customer_churn.csv"]
csv_path = next((DATA_DIR / name for name in csv_names if (DATA_DIR / name).is_file()), None)
if csv_path is None: raise FileNotFoundError("Put the Telco churn CSV in the data folder.")
data = pd.read_csv(csv_path)
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
y = data.pop("Churn").map({"No": 0, "Yes": 1})
X = data.drop(columns=["customerID"], errors="ignore")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def make_preprocessor(frame):
    number_columns = frame.select_dtypes(include="number").columns.tolist()
    text_columns = frame.select_dtypes(exclude="number").columns.tolist()
    return ColumnTransformer([
        ("numbers", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), number_columns),
        ("text", Pipeline([("fill", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), text_columns),
    ], sparse_threshold=0)
def calculate_scores(name, y_true, probability):
    guess = (probability >= 0.5).astype(int)
    return {"Model": name, "Accuracy": accuracy_score(y_true, guess), "Precision": precision_score(y_true, guess, zero_division=0), "Recall": recall_score(y_true, guess, zero_division=0), "F1": f1_score(y_true, guess, zero_division=0), "ROC_AUC": roc_auc_score(y_true, probability)}
print("Training rows:", len(X_train), " Test rows:", len(X_test))

## Optional warm-up: handwritten digits

This small example shows how Keras trains a neural network. It is separate from the customer churn model.

In [ ]:
(digits, labels), (test_digits, test_labels) = tf.keras.datasets.mnist.load_data()
digit_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Rescaling(1 / 255),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
digit_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
digit_model.fit(digits[:10000], labels[:10000], epochs=2, verbose=2)
digit_model.evaluate(test_digits, test_labels, verbose=0)

## 1. Train a Random Forest baseline

A pipeline prepares numeric and text values before fitting the model. The same test customers will be used to compare both models.

In [ ]:
preprocessor = make_preprocessor(X_train)
forest = Pipeline([("prepare", preprocessor), ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1))])
forest.fit(X_train, y_train)
forest_probability = forest.predict_proba(X_test)[:, 1]
forest_scores = calculate_scores("Random Forest", y_test, forest_probability)
pd.Series(forest_scores)
joblib.dump(forest, MODEL_DIR / "day5_notebook_random_forest.joblib")

## 2. Train the ANN

Fit preprocessing only on training rows. The ANN has two small hidden layers and one output that is the probability of churn. Early stopping keeps the best validation result.

In [ ]:
preprocessor = make_preprocessor(X_train)
X_train_ready = preprocessor.fit_transform(X_train).astype("float32")
X_test_ready = preprocessor.transform(X_test).astype("float32")
X_fit, X_valid, y_fit, y_valid = train_test_split(X_train_ready, y_train.to_numpy(), test_size=0.2, random_state=42, stratify=y_train)

ann = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_ready.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = ann.fit(X_fit, y_fit, validation_data=(X_valid, y_valid), epochs=100, batch_size=32, callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)], verbose=2)
ann_probability = ann.predict(X_test_ready, verbose=0).ravel()
ann_scores = calculate_scores("Day 5 ANN", y_test, ann_probability)
comparison = pd.DataFrame([forest_scores, ann_scores])
display(comparison)
ann.save(MODEL_DIR / "day5_notebook_ann.keras")
comparison.to_csv(REPORT_DIR / "day5_notebook_comparison.csv", index=False)

## 3. Check training

The plot shows training and validation loss. If validation loss begins to rise while training loss falls, the model may be overfitting.

In [ ]:
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()